TF-IDF with Cosine Similarity and Price-Based Filtering

Implemented by: Mr. Sundara Pandiyan

In [1]:
from google.colab import files

uploaded = files.upload()

Saving amazon_products_with_images (1).csv to amazon_products_with_images (1).csv


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Load dataset
df = pd.read_csv("/content/amazon_products_with_images (1).csv")

# Missing values handle
df = df.fillna("")


# Combine important features
df["combined_features"] = (
    df["product_name"].astype(str) + " " +
    df["category"].astype(str) + " " +
    df["brand"].astype(str) + " " +
    df["description"].astype(str)
)


# TF-IDF Model
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["combined_features"])


# Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix)


def recommend(product_name, top_n=5):

    product_index = df[
        df["product_name"].str.lower() == product_name.lower()
    ].index


    if len(product_index) == 0:
        return "Product not found"


    index = product_index[0]


    similarity_scores = list(
        enumerate(cosine_sim[index])
    )


    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )


    selected_price = float(df.iloc[index]["price"])


    recommendations = []


    for i, score in similarity_scores[1:]:

        product_price = float(df.iloc[i]["price"])

        product_rating = float(df.iloc[i]["rating"])


        # Similar price range (+/-20%)
        if abs(product_price - selected_price) <= selected_price * 0.20:

            recommendations.append(
                df.iloc[i]
            )


    result = pd.DataFrame(recommendations)


    # Higher rating first
    result = result.sort_values(
        by="rating",
        ascending=False
    )


    return result.head(top_n)



if __name__ == "__main__":

    product = df.iloc[0]["product_name"]

    print("Selected Product:")
    print(product)

    print("\nRecommended Products:")

    print(recommend(product))

Selected Product:
Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)

Recommended Products:
     product_id                                       product_name  \
174  B0BP7XLX48  Syncwire LTG to USB Cable for Fast Charging Co...   
882  B08WLY8V9S  Tukzer Gel Mouse Pad Wrist Rest Memory-Foam Er...   
114  B09TT6BFDX  Cotbolt Silicone Protective Case Cover for LG ...   
70   B0B86CDHL1  oraimo 65W Type C to C Fast Charging Cable USB...   
739  B0752LL57V  Casio MJ-12D 150 Steps Check and Correct Deskt...   

                 category  price  actual_price  discount_percentage  rating  \
174              Charging  399.0        1999.0                 80.0     5.0   
882  Computer Peripherals  425.0         899.0                 53.0     4.5   
114            Television  399.0        1999.0                 80.0     4.5   
70               Charging  349.0         899.0             